# NTO AI Final v2: Multi-ALS + KNN + 3x CatBoostRanker + UserCF + Trends

**Key improvements:** ALS 192/128, 500 candidates, User-CF, item trend features, 3 CatBoost models, 4 folds, cross features

In [2]:
!pip install implicit

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 70.3/70.3 kB 3.3 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for implicit: filename=implicit-0.7.2-cp312-cp312-linux_x86_64.whl size=10793360 sha256=4dd5b127097887f4ee74669d9639e04787cf1640068dc7cb3eeccc28b64b5db1
  Stored in directory: /root/.cache/pip/wheels/b2/00/4f/9ff8af07a0a53ac6007ea5d739da19cfe147a2df542b6899f8
Successfully built implicit


In [ ]:
"""
NTO AI 2025/2026 Final — «Потеряшки» — Full Pipeline
Target: NDCG@20 >= 0.12
Architecture: Multi-source Retrieval + CatBoost Ranking + Stacking + MMR
"""

from __future__ import annotations
import gc, math, os, random, warnings
from collections import defaultdict, Counter
from dataclasses import dataclass
from pathlib import Path
from typing import Any

import numpy as np
import pandas as pd
from scipy import sparse
from sklearn.neighbors import NearestNeighbors
from sklearn.preprocessing import normalize
from tqdm.auto import tqdm

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 200)

# ═══════════════════════════════════════════════════════════════
# 0. CONFIG
# ═══════════════════════════════════════════════════════════════
@dataclass
class CFG:
    seed: int = 42
    use_gpu: bool = Path("/proc/driver/nvidia/version").exists()
    data_root: Path = Path("/kaggle/input/datasets/artemnazemtsev/nto-ai")
    output_dir: Path = Path("/kaggle/working")

    # Validation
    n_pseudo_folds: int = 4
    fold_days: int = 30
    hide_rate: float = 0.20

    # Weights
    read_weight: float = 2.5
    wishlist_weight: float = 1.0
    full_tau: float = 90.0
    recent_tau: float = 21.0

    # ALS
    als_factors: int = 192
    als_reg: float = 0.05
    als_iter: int = 16
    als_rec_factors: int = 128
    als_rec_iter: int = 12
    als_extra_factors: int = 256
    als_extra_reg: float = 0.01
    als_extra_iter: int = 14

    # Candidates
    als_topk: int = 200
    als_rec_topk: int = 150
    als_extra_topk: int = 120
    knn_latent_k: int = 100
    knn_sparse_k: int = 80
    cooc_topk: int = 60
    ucf_neighbors: int = 30
    ucf_items: int = 15
    seed_items_recent: int = 8
    seed_items_heavy: int = 8
    max_cands: int = 700
    top_authors: int = 8
    top_books: int = 10
    top_genres: int = 8
    top_entity_items: int = 25
    top_demo: int = 25
    top_global: int = 40

    # CatBoost
    cb_iter: int = 2500
    cb_lr: float = 0.04
    cb_iter_b: int = 2500
    cb_lr_b: float = 0.035
    cb_iter_c: int = 2000
    cb_lr_c: float = 0.04

    # Ranking
    max_neg: int = 220
    min_group: int = 8


cfg = CFG()
if not cfg.data_root.exists():
    # Try alternative paths
    for p in [Path("/kaggle/input"), Path(".")]:
        if not p.exists():
            continue
        for pp in p.rglob("interactions.csv"):
            cfg.data_root = pp.parent
            break
    print("Data root:", cfg.data_root)
else:
    print("Data root:", cfg.data_root)

cfg.output_dir.mkdir(parents=True, exist_ok=True)
random.seed(cfg.seed)
np.random.seed(cfg.seed)
os.environ["PYTHONHASHSEED"] = str(cfg.seed)
TOP_K = 20

# ═══════════════════════════════════════════════════════════════
# 1. LOAD DATA
# ═══════════════════════════════════════════════════════════════
print("Loading data...")
interactions = pd.read_csv(cfg.data_root / "interactions.csv", low_memory=False)
targets = pd.read_csv(cfg.data_root / "targets.csv")
editions = pd.read_csv(cfg.data_root / "editions.csv", low_memory=False)
users_df = pd.read_csv(cfg.data_root / "users.csv")
authors = pd.read_csv(cfg.data_root / "authors.csv", low_memory=False)
genres = pd.read_csv(cfg.data_root / "genres.csv", low_memory=False)
book_genres = pd.read_csv(cfg.data_root / "book_genres.csv")

interactions["event_ts"] = pd.to_datetime(interactions["event_ts"])
interactions = interactions[interactions["event_type"].isin([1, 2])].copy()
for c in ["user_id", "edition_id"]:
    interactions[c] = interactions[c].astype("int64")
interactions["event_type"] = interactions["event_type"].astype("int8")
interactions["rating"] = pd.to_numeric(interactions["rating"], errors="coerce").astype("float32")
interactions = interactions.sort_values("event_ts").reset_index(drop=True)

users_df["user_id"] = users_df["user_id"].astype("int64")
users_df["gender"] = pd.to_numeric(users_df["gender"], errors="coerce").fillna(-1).astype("int16")
users_df["age"] = pd.to_numeric(users_df["age"], errors="coerce").astype("float32")
bins = [-1, 17, 24, 34, 44, 54, 120]
users_df["age_bucket"] = pd.cut(users_df["age"].fillna(-1), bins=bins, labels=[0,1,2,3,4,5]).astype("float").fillna(-1).astype("int16")
targets["user_id"] = targets["user_id"].astype("int64")

editions["edition_id"] = editions["edition_id"].astype("int64")
editions["book_id"] = pd.to_numeric(editions["book_id"], errors="coerce").fillna(-1).astype("int64")
editions["author_id"] = pd.to_numeric(editions["author_id"], errors="coerce").fillna(-1).astype("int64")
editions["publication_year"] = pd.to_numeric(editions["publication_year"], errors="coerce").astype("float32")
editions["age_restriction"] = pd.to_numeric(editions["age_restriction"], errors="coerce").fillna(-1).astype("int16")
editions["language_id"] = pd.to_numeric(editions["language_id"], errors="coerce").fillna(-1).astype("int64")
editions["publisher_id"] = pd.to_numeric(editions["publisher_id"], errors="coerce").fillna(-1).astype("int64")
for c in ["title", "description"]:
    if c not in editions.columns:
        editions[c] = ""
    editions[c] = editions[c].fillna("").astype(str)
editions["title_len"] = editions["title"].str.len().astype("int32")
editions["desc_len"] = editions["description"].str.len().astype("int32")

authors["author_id"] = authors["author_id"].astype("int64")
book_genres["book_id"] = pd.to_numeric(book_genres["book_id"], errors="coerce").fillna(-1).astype("int64")
book_genres["genre_id"] = pd.to_numeric(book_genres["genre_id"], errors="coerce").fillna(-1).astype("int64")

# Edition meta
genre_counts = book_genres.groupby("book_id")["genre_id"].nunique().reset_index()
genre_counts.columns = ["book_id", "genre_count"]
edition_meta = editions.merge(genre_counts, on="book_id", how="left")
edition_meta["genre_count"] = edition_meta["genre_count"].fillna(0).astype("int16")
edition_meta["item_age_proxy"] = (2025 - edition_meta["publication_year"]).clip(-100, 300).fillna(0).astype("float32")
edition_meta["title_desc_len"] = (edition_meta["title_len"] + edition_meta["desc_len"]).astype("int32")

# Edition genres lookup
edition_genres = editions[["edition_id", "book_id"]].merge(book_genres, on="book_id", how="left")
edition_genres = edition_genres[edition_genres["genre_id"].notna() & (edition_genres["genre_id"] >= 0)].copy()
edition_genres["genre_id"] = edition_genres["genre_id"].astype("int64")

for k, v in {"interactions": interactions, "targets": targets, "editions": editions, "users": users_df}.items():
    print(f"  {k}: {v.shape}")

# ═══════════════════════════════════════════════════════════════
# 2. HELPERS
# ═══════════════════════════════════════════════════════════════
def ndcg_at_k(predicted, relevant, k=TOP_K):
    if not relevant: return 0.0
    dcg = sum(1.0 / math.log2(i + 2) for i, item in enumerate(predicted[:k]) if item in relevant)
    idcg = sum(1.0 / math.log2(i + 2) for i in range(min(len(relevant), k)))
    return dcg / idcg if idcg > 0 else 0.0

def eval_scored(scored_df, hidden_pairs, score_col, k=TOP_K):
    rel = hidden_pairs.groupby("user_id")["edition_id"].agg(lambda x: set(int(v) for v in x)).to_dict()
    total, cnt = 0.0, 0
    for uid, grp in scored_df.sort_values(["user_id", score_col], ascending=[True, False]).groupby("user_id"):
        r = rel.get(int(uid), set())
        if not r: continue
        total += ndcg_at_k(grp["edition_id"].head(k).astype(int).tolist(), r, k)
        cnt += 1
    return total / max(cnt, 1)

def recall_at_k(cands, hidden, k=200):
    rel = hidden.assign(label=1)
    topk = cands.sort_values(["user_id", "pre_score"], ascending=[True, False]).groupby("user_id").head(k)[["user_id", "edition_id"]].drop_duplicates()
    m = rel.merge(topk, on=["user_id", "edition_id"], how="left", indicator=True)
    return float((m["_merge"] == "both").mean()) if len(m) else 0.0

# ═══════════════════════════════════════════════════════════════
# 3. INTERACTION WEIGHTS & PAIR AGGREGATES
# ═══════════════════════════════════════════════════════════════
def weight_interactions(df, ref_ts):
    df = df.copy()
    age = ((ref_ts - df["event_ts"]).dt.total_seconds() / 86400.0).clip(lower=0)
    evt_w = np.where(df["event_type"] == 2, cfg.read_weight, cfg.wishlist_weight).astype("float32")
    rat = np.where(df["rating"].notna(), 1.0 + 0.08 * (df["rating"].fillna(0).values - 3.0), 1.0).astype("float32")
    rat = np.clip(rat, 0.65, 1.35)
    df["is_read"] = (df["event_type"] == 2).astype("int8")
    df["is_wish"] = (df["event_type"] == 1).astype("int8")
    df["w_full"] = evt_w * (0.35 + 0.65 * np.exp(-age / cfg.full_tau)) * rat
    df["w_recent"] = evt_w * np.exp(-age / cfg.recent_tau) * rat
    df["age_days"] = age.astype("float32")
    return df

def build_pairs(obs, ref_ts):
    w = weight_interactions(obs, ref_ts)
    p = w.groupby(["user_id", "edition_id"], as_index=False).agg(
        n=("event_ts", "size"), reads=("is_read", "sum"), wish=("is_wish", "sum"),
        w_full=("w_full", "sum"), w_recent=("w_recent", "sum"),
        last_ts=("event_ts", "max"), first_ts=("event_ts", "min"),
        max_rat=("rating", "max"), mean_rat=("rating", "mean"),
    )
    p["last_days"] = ((ref_ts - p["last_ts"]).dt.total_seconds() / 86400).astype("float32")
    p["first_days"] = ((ref_ts - p["first_ts"]).dt.total_seconds() / 86400).astype("float32")
    return p

# ═══════════════════════════════════════════════════════════════
# 4. ALS MODELS
# ═══════════════════════════════════════════════════════════════
try:
    import implicit
    HAS_IMPLICIT = True
except:
    HAS_IMPLICIT = False
print("implicit:", HAS_IMPLICIT)

def build_matrix(pair_df, val_col):
    uids = np.sort(pair_df["user_id"].unique())
    iids = np.sort(pair_df["edition_id"].unique())
    u2i = {int(u): i for i, u in enumerate(uids)}
    i2i = {int(it): i for i, it in enumerate(iids)}
    r = pair_df["user_id"].map(u2i).values
    c = pair_df["edition_id"].map(i2i).values
    v = pair_df[val_col].values.astype(np.float32)
    mat = sparse.csr_matrix((v, (r, c)), shape=(len(uids), len(iids)))
    mat.sum_duplicates()
    return mat, u2i, iids

def norm_rows(x, eps=1e-12):
    d = np.linalg.norm(x, axis=1, keepdims=True)
    return x / np.where(d < eps, 1, d)

class ALSWrap:
    def __init__(self, factors, reg, iters, seed, gpu):
        self.factors, self.reg, self.iters, self.seed, self.gpu = factors, reg, iters, seed, gpu
        self.u2i = {}; self.i2i = {}; self.idx2item = None; self.seen = None
        self.uf = None; self.itf = None; self.itf_n = None; self.backend = ""

    def fit(self, pair_df, val_col):
        mat, self.u2i, self.idx2item = build_matrix(pair_df, val_col)
        self.i2i = {int(it): i for i, it in enumerate(self.idx2item)}
        self.seen = mat.tocsr()
        if HAS_IMPLICIT:
            try:
                m = implicit.als.AlternatingLeastSquares(
                    factors=self.factors, regularization=self.reg,
                    iterations=self.iters, random_state=self.seed, use_gpu=self.gpu)
                # implicit >=0.6 expects user_items; older expects item_users
                # Try user_items first (mat), fall back to mat.T
                n_users, n_items = mat.shape
                try:
                    m.fit(mat, show_progress=True)
                except Exception:
                    m.fit(mat.T.tocsr(), show_progress=True)
                uf, itf = m.user_factors, m.item_factors
                if hasattr(uf, "to_numpy"): uf = uf.to_numpy()
                if hasattr(itf, "to_numpy"): itf = itf.to_numpy()
                uf = np.array(uf, dtype=np.float32)
                itf = np.array(itf, dtype=np.float32)
                # Auto-detect swapped factors by checking shapes
                if uf.shape[0] == n_items and itf.shape[0] == n_users:
                    uf, itf = itf, uf  # swap
                elif uf.shape[0] != n_users or itf.shape[0] != n_items:
                    # Shapes don't match either way - take best guess
                    if uf.shape[0] == n_items:
                        uf, itf = itf, uf
                self.uf = uf
                self.itf = itf
                self.backend = "implicit"
                print(f"  ALS fitted: users={self.uf.shape}, items={self.itf.shape}, matrix={mat.shape}")
            except Exception as e:
                print(f"ALS fallback: {e}")
                self._fallback(mat)
        else:
            self._fallback(mat)
        self.itf_n = norm_rows(self.itf)
        return self

    def _fallback(self, mat):
        self.backend = "fallback"
        f = min(self.factors, 64); it = min(self.iters, 8)
        rng = np.random.default_rng(self.seed)
        self.uf = (0.01 * rng.standard_normal((mat.shape[0], f))).astype(np.float32)
        self.itf = (0.01 * rng.standard_normal((mat.shape[1], f))).astype(np.float32)
        eye = np.eye(f, dtype=np.float32)
        Ciu = mat.T.tocsr()
        for _ in range(it):
            self.uf = self._solve(mat, self.itf, eye)
            self.itf = self._solve(Ciu, self.uf, eye)

    def _solve(self, C, Y, eye):
        X = np.zeros((C.shape[0], Y.shape[1]), dtype=np.float32)
        YtY = Y.T @ Y
        for u in range(C.shape[0]):
            s, e = C.indptr[u], C.indptr[u+1]
            idx, conf = C.indices[s:e], C.data[s:e]
            if len(idx) == 0: continue
            A = YtY + self.reg * eye
            b = np.zeros(Y.shape[1], dtype=np.float32)
            for i, c in zip(idx, conf):
                y = Y[i]; A += (c-1) * np.outer(y, y); b += c * y
            X[u] = np.linalg.solve(A, b)
        return X

    def score_pairs(self, uids, iids):
        sc = np.zeros(len(uids), dtype=np.float32)
        cos = np.zeros(len(uids), dtype=np.float32)
        if self.uf is None: return sc, cos
        mask = np.array([int(u) in self.u2i and int(i) in self.i2i for u, i in zip(uids, iids)])
        if mask.any():
            ui = np.array([self.u2i[int(u)] for u in uids[mask]])
            ii = np.array([self.i2i[int(i)] for i in iids[mask]])
            uv, iv = self.uf[ui], self.itf[ii]
            sc[mask] = np.sum(uv * iv, axis=1)
            un, ine = np.linalg.norm(uv, axis=1), np.linalg.norm(iv, axis=1)
            cos[mask] = sc[mask] / np.maximum(un * ine, 1e-12)
        return sc.astype(np.float32), cos.astype(np.float32)

    def recommend(self, user_ids, n, bs=128):
        rows = []
        valid = [int(u) for u in user_ids if int(u) in self.u2i]
        if not valid: return pd.DataFrame(columns=["user_id", "edition_id", "rank", "score"])
        for s in range(0, len(valid), bs):
            batch = valid[s:s+bs]
            bi = np.array([self.u2i[u] for u in batch])
            scores = self.uf[bi] @ self.itf.T
            n_score_items = scores.shape[1]
            for ri, uid in enumerate(batch):
                seen_idx = self.seen[bi[ri]].indices
                seen_idx = seen_idx[seen_idx < n_score_items]  # bounds safety
                scores[ri, seen_idx] = -1e18
            k = min(n, scores.shape[1])
            if k <= 0: continue
            top = np.argpartition(-scores, min(k-1, scores.shape[1]-1), axis=1)[:, :k]
            for ri, uid in enumerate(batch):
                idx = top[ri]; sc = scores[ri, idx]; order = np.argsort(-sc)
                idx, sc = idx[order], sc[order]
                rows.append(pd.DataFrame({"user_id": uid, "edition_id": self.idx2item[idx].astype(np.int64),
                    "rank": np.arange(1, len(idx)+1, dtype=np.int32), "score": sc.astype(np.float32)}))
        return pd.concat(rows, ignore_index=True) if rows else pd.DataFrame(columns=["user_id", "edition_id", "rank", "score"])

def fit_bpr(pair_df, cfg):
    if not HAS_IMPLICIT: return None
    try:
        mat, u2i, idx2item = build_matrix(pair_df, "w_full")
        n_users, n_items = mat.shape
        m = implicit.bpr.BayesianPersonalizedRanking(
            factors=128, regularization=0.01, iterations=100,
            random_state=cfg.seed + 99, use_gpu=cfg.use_gpu)
        try:
            m.fit(mat, show_progress=True)
        except Exception:
            m.fit(mat.T.tocsr(), show_progress=True)
        wrap = ALSWrap(128, 0.01, 100, cfg.seed+99, cfg.use_gpu)
        wrap.u2i, wrap.idx2item, wrap.i2i = u2i, idx2item, {int(it): i for i, it in enumerate(idx2item)}
        wrap.seen = mat.tocsr()
        uf, itf = m.user_factors, m.item_factors
        if hasattr(uf, "to_numpy"): uf = uf.to_numpy()
        if hasattr(itf, "to_numpy"): itf = itf.to_numpy()
        uf = np.array(uf, dtype=np.float32)
        itf = np.array(itf, dtype=np.float32)
        if uf.shape[0] == n_items and itf.shape[0] == n_users:
            uf, itf = itf, uf
        wrap.uf = uf
        wrap.itf = itf
        wrap.itf_n = norm_rows(wrap.itf)
        wrap.backend = "bpr"
        print("BPR OK")
        return wrap
    except Exception as e:
        print(f"BPR skip: {e}")
        return None

# ═══════════════════════════════════════════════════════════════
# 5. KNN + CO-OCCURRENCE + USER-CF
# ═══════════════════════════════════════════════════════════════
def fit_knn(pair_df, als):
    lat_nn = None
    if als.itf_n is not None and len(als.itf_n) > 1:
        lat_nn = NearestNeighbors(metric="cosine", algorithm="auto").fit(als.itf_n)
    mat, _, idx2item = build_matrix(pair_df, "w_recent")
    iu = normalize(mat.T.tocsr(), norm="l2", axis=1)
    sp_nn = NearestNeighbors(metric="cosine", algorithm="brute").fit(iu) if iu.shape[0] > 1 else None
    sp_i2i = {int(it): i for i, it in enumerate(idx2item)}
    # User NN
    u_nn = None
    if als.uf is not None and len(als.uf) > 1:
        u_nn = NearestNeighbors(metric="cosine", algorithm="auto").fit(norm_rows(als.uf))
    return {"lat_nn": lat_nn, "sp_nn": sp_nn, "sp_iu": iu, "sp_idx": idx2item, "sp_i2i": sp_i2i, "u_nn": u_nn}

def query_nbrs(als, seeds, nn, topk, mode="latent"):
    if mode == "latent":
        if nn is None or als.itf_n is None: return pd.DataFrame(columns=["seed_item_id","edition_id","nbr_rank","sim"])
        valid = [it for it in np.unique(seeds) if int(it) in als.i2i]
        if not valid: return pd.DataFrame(columns=["seed_item_id","edition_id","nbr_rank","sim"])
        idx = np.array([als.i2i[int(it)] for it in valid])
        k = min(topk+1, als.itf_n.shape[0])
        d, ind = nn.kneighbors(als.itf_n[idx], n_neighbors=k)
    else:  # sparse
        return pd.DataFrame(columns=["seed_item_id","edition_id","nbr_rank","sim"])  # handled separately
    rows = []
    for si, dr, ir in zip(valid, d, ind):
        sim = 1.0 - dr
        items = als.idx2item[ir].astype(np.int64)
        df = pd.DataFrame({"seed_item_id": int(si), "edition_id": items, "sim": sim.astype(np.float32)})
        df = df[df["edition_id"] != int(si)].head(topk)
        df["nbr_rank"] = np.arange(1, len(df)+1, dtype=np.int32)
        rows.append(df)
    return pd.concat(rows, ignore_index=True) if rows else pd.DataFrame(columns=["seed_item_id","edition_id","nbr_rank","sim"])

def query_sparse_nbrs(seeds, knn_info, topk):
    nn = knn_info["sp_nn"]
    if nn is None: return pd.DataFrame(columns=["seed_item_id","edition_id","nbr_rank","sim"])
    valid = [it for it in np.unique(seeds) if int(it) in knn_info["sp_i2i"]]
    if not valid: return pd.DataFrame(columns=["seed_item_id","edition_id","nbr_rank","sim"])
    idx = np.array([knn_info["sp_i2i"][int(it)] for it in valid])
    k = min(topk+1, knn_info["sp_iu"].shape[0])
    d, ind = nn.kneighbors(knn_info["sp_iu"][idx], n_neighbors=k)
    rows = []
    for si, dr, ir in zip(valid, d, ind):
        sim = 1.0 - dr; items = knn_info["sp_idx"][ir].astype(np.int64)
        df = pd.DataFrame({"seed_item_id": int(si), "edition_id": items, "sim": sim.astype(np.float32)})
        df = df[df["edition_id"] != int(si)].head(topk)
        df["nbr_rank"] = np.arange(1, len(df)+1, dtype=np.int32)
        rows.append(df)
    return pd.concat(rows, ignore_index=True) if rows else pd.DataFrame(columns=["seed_item_id","edition_id","nbr_rank","sim"])

def build_cooc_cands(pair_df, seeds_df, user_ids, cfg):
    """Fast co-occurrence via sparse matrix multiplication."""
    uids_arr = np.sort(pair_df["user_id"].unique())
    iids_arr = np.sort(pair_df["edition_id"].unique())
    u2i = {int(u): i for i, u in enumerate(uids_arr)}
    i2i = {int(it): i for i, it in enumerate(iids_arr)}
    r = pair_df["user_id"].map(u2i).values
    c = pair_df["edition_id"].map(i2i).values
    ui = sparse.csr_matrix((np.ones(len(r), dtype=np.float32), (r, c)), shape=(len(uids_arr), len(iids_arr)))
    rows = []
    seed_sub = seeds_df[seeds_df["user_id"].isin(user_ids)]
    for uid in seed_sub["user_id"].unique():
        if int(uid) not in u2i: continue
        us = seed_sub[seed_sub["user_id"] == uid]
        si, sw = [], []
        for _, row in us.iterrows():
            eid = int(row["edition_id"])
            if eid in i2i: si.append(i2i[eid]); sw.append(float(row["seed_w"]))
        if not si: continue
        sv = sparse.csr_matrix((sw, ([0]*len(si), si)), shape=(1, len(iids_arr)))
        user_sc = ui @ sv.T
        user_sc[u2i[int(uid)], 0] = 0
        item_sc = ui.T @ user_sc
        arr = np.asarray(item_sc.todense()).flatten()
        for s in si: arr[s] = 0
        seen_cols = ui[u2i[int(uid)]].indices
        arr[seen_cols] = 0
        tk = cfg.cooc_topk * 3
        if len(arr) > tk:
            top = np.argpartition(-arr, tk)[:tk]
        else:
            top = np.arange(len(arr))
        top = top[arr[top] > 0]
        top = top[np.argsort(-arr[top])]
        if len(top) == 0: continue
        df = pd.DataFrame({"edition_id": iids_arr[top].astype(np.int64), "cooc_score": arr[top].astype(np.float32)})
        df["user_id"] = int(uid)
        df["cooc_rank"] = np.arange(1, len(df)+1, dtype=np.int32)
        rows.append(df[["user_id","edition_id","cooc_score","cooc_rank"]])
    return pd.concat(rows, ignore_index=True) if rows else pd.DataFrame(columns=["user_id","edition_id","cooc_score","cooc_rank"])

def build_ucf_cands(als, pair_df, knn_info, user_ids, cfg):
    u_nn = knn_info.get("u_nn")
    if u_nn is None or als.uf is None:
        return pd.DataFrame(columns=["user_id","edition_id","ucf_score","ucf_rank"])
    valid = [int(u) for u in user_ids if int(u) in als.u2i]
    if not valid: return pd.DataFrame(columns=["user_id","edition_id","ucf_score","ucf_rank"])
    uf_n = norm_rows(als.uf)
    ui = np.array([als.u2i[u] for u in valid])
    k = min(cfg.ucf_neighbors + 1, als.uf.shape[0])
    d, ind = u_nn.kneighbors(uf_n[ui], n_neighbors=k)
    utop = pair_df.sort_values(["user_id","w_recent"], ascending=[True,False]).groupby("user_id").head(cfg.ucf_items)
    i2u = {v: k for k, v in als.u2i.items()}
    rows = []
    for i, uid in enumerate(valid):
        sim = 1.0 - d[i]; ni = ind[i]
        mask = ni != als.u2i[uid]; sim, ni = sim[mask], ni[mask]
        nuids = [i2u.get(int(n)) for n in ni]; nuids = [u for u in nuids if u is not None]
        if not nuids: continue
        items = utop[utop["user_id"].isin(nuids)][["user_id","edition_id","w_recent"]].copy()
        if items.empty: continue
        sm = {u: s for u, s in zip(nuids, sim[:len(nuids)])}
        items["_s"] = items["user_id"].map(sm).fillna(0).astype("float32")
        items["_sc"] = items["_s"] * np.log1p(items["w_recent"])
        agg = items.groupby("edition_id", as_index=False).agg(ucf_score=("_sc","sum"))
        agg = agg.sort_values("ucf_score", ascending=False).head(cfg.ucf_items * 3)
        agg["user_id"] = uid; agg["ucf_rank"] = np.arange(1, len(agg)+1, dtype=np.int32)
        rows.append(agg[["user_id","edition_id","ucf_score","ucf_rank"]])
    return pd.concat(rows, ignore_index=True) if rows else pd.DataFrame(columns=["user_id","edition_id","ucf_score","ucf_rank"])

# ═══════════════════════════════════════════════════════════════
# 6. CONTEXT BUILDER (full pipeline for one time slice)
# ═══════════════════════════════════════════════════════════════
def r2s(series, shift=1.0):
    return 1.0 / (series.astype(float) + shift)

def build_context(obs, ref_ts):
    """Build all retrieval structures from observed interactions up to ref_ts."""
    obs_w = weight_interactions(obs, ref_ts)
    pair = build_pairs(obs, ref_ts)

    # User features
    uf = pair.groupby("user_id", as_index=False).agg(
        u_items=("edition_id","nunique"), u_events=("n","sum"), u_reads=("reads","sum"),
        u_wish=("wish","sum"), u_w=("w_full","sum"), u_wr=("w_recent","sum"),
        u_last=("last_days","min"), u_avg_rat=("mean_rat","mean"))
    for d in [7,14,30,90]:
        tmp = obs_w[obs_w["event_ts"] >= ref_ts - pd.Timedelta(days=d)]
        if tmp.empty: t2 = pd.DataFrame(columns=["user_id"])
        else: t2 = tmp.groupby("user_id", as_index=False).agg(**{f"u_ev{d}":("event_ts","size"), f"u_it{d}":("edition_id","nunique"), f"u_w{d}":("w_full","sum")})
        uf = uf.merge(t2, on="user_id", how="left")
    uf = uf.merge(users_df[["user_id","gender","age","age_bucket"]], on="user_id", how="left")
    for c in uf.columns:
        if c != "user_id": uf[c] = uf[c].fillna(0 if "age" not in c and "gender" not in c else -1)
    uf["u_read_sh"] = (uf["u_reads"] / np.maximum(uf["u_events"], 1)).astype("float32")
    uf["u_rec_ratio"] = (uf["u_wr"] / np.maximum(uf["u_w"], 1e-6)).astype("float32")
    uf["u_trend"] = (uf.get("u_w14", pd.Series(0, index=uf.index)).fillna(0) / np.maximum(uf.get("u_w30", pd.Series(1, index=uf.index)).fillna(1), 1e-6)).astype("float32")

    # Item features
    itf = pair.groupby("edition_id", as_index=False).agg(
        i_users=("user_id","nunique"), i_events=("n","sum"), i_reads=("reads","sum"),
        i_wish=("wish","sum"), i_w=("w_full","sum"), i_wr=("w_recent","sum"),
        i_last=("last_days","min"), i_avg_rat=("mean_rat","mean"))
    for d in [7,14,30,90]:
        tmp = obs_w[obs_w["event_ts"] >= ref_ts - pd.Timedelta(days=d)]
        if tmp.empty: t2 = pd.DataFrame(columns=["edition_id"])
        else: t2 = tmp.groupby("edition_id", as_index=False).agg(**{f"i_ev{d}":("event_ts","size"), f"i_us{d}":("user_id","nunique"), f"i_w{d}":("w_full","sum")})
        itf = itf.merge(t2, on="edition_id", how="left")
    itf = itf.merge(edition_meta[["edition_id","book_id","author_id","publication_year","age_restriction",
        "language_id","publisher_id","title_len","desc_len","genre_count","item_age_proxy","title_desc_len"]], on="edition_id", how="left")
    for c in itf.columns:
        if c not in ("edition_id",): itf[c] = itf[c].fillna(0)
    itf["i_read_sh"] = (itf["i_reads"] / np.maximum(itf["i_events"], 1)).astype("float32")
    itf["i_rec_ratio"] = (itf["i_wr"] / np.maximum(itf["i_w"], 1e-6)).astype("float32")
    itf["i_trend7"] = (itf.get("i_w7", pd.Series(0)).fillna(0) / np.maximum(itf.get("i_w30", pd.Series(1)).fillna(1), 1e-6)).astype("float32")
    itf["i_trend14"] = (itf.get("i_w14", pd.Series(0)).fillna(0) / np.maximum(itf.get("i_w30", pd.Series(1)).fillna(1), 1e-6)).astype("float32")
    itf["i_hot"] = (itf.get("i_w7", pd.Series(0)).fillna(0) / np.maximum(itf["i_w"], 1e-6)).astype("float32")
    itf["i_pop_rank"] = itf["i_wr"].rank(method="dense", ascending=False).astype("float32")

    # Affinity
    pm = pair.merge(edition_meta[["edition_id","author_id","book_id","language_id","publisher_id"]], on="edition_id", how="left")
    def _agg(col, pre):
        o = pm.groupby(["user_id", col], as_index=False).agg(**{
            f"{pre}_w":("w_full","sum"), f"{pre}_wr":("w_recent","sum"),
            f"{pre}_n":("edition_id","nunique"), f"{pre}_last":("last_days","min")})
        o[f"{pre}_rank"] = o.groupby("user_id")[f"{pre}_wr"].rank(method="dense", ascending=False).astype("float32")
        return o
    ua = _agg("author_id","ua"); ub = _agg("book_id","ub"); ul = _agg("language_id","ul"); up = _agg("publisher_id","up")
    pg = pair[["user_id","edition_id","w_full","w_recent","last_days"]].merge(edition_genres, on="edition_id", how="left")
    pg = pg[pg["genre_id"].notna()]; pg["genre_id"] = pg["genre_id"].astype("int64")
    ug = pg.groupby(["user_id","genre_id"], as_index=False).agg(ug_w=("w_full","sum"), ug_wr=("w_recent","sum"), ug_n=("edition_id","nunique"), ug_last=("last_days","min"))
    ug["ug_rank"] = ug.groupby("user_id")["ug_wr"].rank(method="dense", ascending=False).astype("float32")

    # Seeds
    rec = pair.sort_values(["user_id","last_days","w_recent"], ascending=[True,True,False]).groupby("user_id").head(cfg.seed_items_recent).copy()
    rec["seed_w"] = (1.0 / (1 + rec["last_days"]/7) * (1 + 0.15*rec["reads"])).astype("float32")
    hvy = pair.sort_values(["user_id","w_recent","w_full"], ascending=[True,False,False]).groupby("user_id").head(cfg.seed_items_heavy).copy()
    hvy["seed_w"] = np.log1p(hvy["w_recent"] + hvy["w_full"]).astype("float32")
    seeds = pd.concat([rec[["user_id","edition_id","seed_w","last_days","reads"]],
                        hvy[["user_id","edition_id","seed_w","last_days","reads"]]], ignore_index=True)
    seeds = seeds.groupby(["user_id","edition_id"], as_index=False).agg(seed_w=("seed_w","max"), last_days=("last_days","min"), reads=("reads","max"))

    # Popularity tables
    ipop = pair.groupby("edition_id", as_index=False).agg(pop=("w_recent","sum"), pop_full=("w_full","sum"), pop_users=("user_id","nunique"))
    ipop["pop_rank"] = ipop["pop"].rank(method="dense", ascending=False)
    gpop = ipop.sort_values("pop", ascending=False).head(cfg.top_global)

    # Author/book/genre pop
    aipop = pm.groupby(["author_id","edition_id"], as_index=False).agg(ai_sc=("w_recent","sum"))
    aipop["ai_rank"] = aipop.groupby("author_id")["ai_sc"].rank(method="dense", ascending=False).astype("float32")
    aipop = aipop[aipop["ai_rank"] <= cfg.top_entity_items]
    bipop = pm.groupby(["book_id","edition_id"], as_index=False).agg(bi_sc=("w_recent","sum"))
    bipop["bi_rank"] = bipop.groupby("book_id")["bi_sc"].rank(method="dense", ascending=False).astype("float32")
    bipop = bipop[bipop["bi_rank"] <= 15]
    gipop = pg.groupby(["genre_id","edition_id"], as_index=False).agg(gi_sc=("w_recent","sum"))
    gipop["gi_rank"] = gipop.groupby("genre_id")["gi_sc"].rank(method="dense", ascending=False).astype("float32")
    gipop = gipop[gipop["gi_rank"] <= cfg.top_entity_items]

    # Demo pop
    ou = pair[["user_id"]].drop_duplicates().merge(users_df[["user_id","gender","age_bucket"]], on="user_id", how="left")
    uir = pair.merge(ou, on="user_id", how="left")
    dpop = uir.groupby(["gender","age_bucket","edition_id"], as_index=False).agg(d_sc=("w_recent","sum"))
    dpop["d_rank"] = dpop.groupby(["gender","age_bucket"])["d_sc"].rank(method="dense", ascending=False).astype("float32")
    dpop = dpop[dpop["d_rank"] <= cfg.top_demo]
    genp = uir.groupby(["gender","edition_id"], as_index=False).agg(g_sc=("w_recent","sum"))
    genp["g_rank"] = genp.groupby("gender")["g_sc"].rank(method="dense", ascending=False).astype("float32")
    genp = genp[genp["g_rank"] <= cfg.top_demo]

    # ALS models
    als1 = ALSWrap(cfg.als_factors, cfg.als_reg, cfg.als_iter, cfg.seed, cfg.use_gpu).fit(pair, "w_full")
    als2 = ALSWrap(cfg.als_rec_factors, cfg.als_reg, cfg.als_rec_iter, cfg.seed+7, cfg.use_gpu).fit(pair, "w_recent")
    als3 = ALSWrap(cfg.als_extra_factors, cfg.als_extra_reg, cfg.als_extra_iter, cfg.seed+13, cfg.use_gpu).fit(pair, "w_full")
    bpr = fit_bpr(pair, cfg)
    knn = fit_knn(pair, als1)
    seen = pair[["user_id","edition_id"]].drop_duplicates()

    return {"ref": ref_ts, "obs_w": obs_w, "pair": pair, "uf": uf, "itf": itf,
        "ua": ua, "ub": ub, "ul": ul, "up": up, "ug": ug, "pm": pm,
        "seeds": seeds, "gpop": gpop, "ipop": ipop, "aipop": aipop, "bipop": bipop, "gipop": gipop,
        "dpop": dpop, "genp": genp,
        "ua_top": ua[ua["ua_rank"] <= cfg.top_authors], "ub_top": ub[ub["ub_rank"] <= cfg.top_books],
        "ug_top": ug[ug["ug_rank"] <= cfg.top_genres],
        "als": {"full": als1, "rec": als2, "extra": als3, **({"bpr": bpr} if bpr else {})},
        "knn": knn, "seen": seen}

# ═══════════════════════════════════════════════════════════════
# 7. CANDIDATE GENERATION
# ═══════════════════════════════════════════════════════════════
def gen_candidates(ctx, user_ids):
    srcs = {}
    # ALS
    for key, tk, pre in [("full", cfg.als_topk, "af"), ("rec", cfg.als_rec_topk, "ar"), ("extra", cfg.als_extra_topk, "ax")]:
        df = ctx["als"][key].recommend(user_ids, tk)
        if not df.empty: df = df.rename(columns={"rank": f"{pre}_rank", "score": f"{pre}_score"})
        srcs[pre] = df
    if "bpr" in ctx["als"]:
        df = ctx["als"]["bpr"].recommend(user_ids, 100)
        if not df.empty: df = df.rename(columns={"rank": "bpr_rank", "score": "bpr_score"})
        srcs["bpr"] = df

    # KNN from seeds
    seeds = ctx["seeds"][ctx["seeds"]["user_id"].isin(user_ids)]
    for mode, nn_key, topk, pre in [("latent", "lat_nn", cfg.knn_latent_k, "lk"), ("sparse", "sp_nn", cfg.knn_sparse_k, "sk")]:
        if mode == "latent":
            nbr = query_nbrs(ctx["als"]["full"], seeds["edition_id"].values, ctx["knn"]["lat_nn"], topk)
        else:
            nbr = query_sparse_nbrs(seeds["edition_id"].values, ctx["knn"], topk)
        if nbr.empty: srcs[pre] = pd.DataFrame(columns=["user_id","edition_id"]); continue
        mg = seeds[["user_id","edition_id","seed_w"]].rename(columns={"edition_id":"seed_item_id"}).merge(nbr, on="seed_item_id", how="inner")
        mg[f"{pre}_ws"] = (mg["seed_w"] * mg["sim"]).astype("float32")
        agg = mg.groupby(["user_id","edition_id"], as_index=False).agg(**{
            f"{pre}_score":(f"{pre}_ws","sum"), f"{pre}_max_sim":("sim","max"),
            f"{pre}_cnt":("sim","size"), f"{pre}_rank":("nbr_rank","min")})
        srcs[pre] = agg

    # Co-occurrence
    srcs["cooc"] = build_cooc_cands(ctx["pair"], ctx["seeds"], user_ids, cfg)
    # User-CF
    srcs["ucf"] = build_ucf_cands(ctx["als"]["full"], ctx["pair"], ctx["knn"], user_ids, cfg)

    # Entity pop
    for ent, etop, eipop, ecol, ewcol, escol, ercol, pre in [
        ("author", ctx["ua_top"], ctx["aipop"], "author_id", "ua_wr", "ai_sc", "ai_rank", "au"),
        ("book", ctx["ub_top"], ctx["bipop"], "book_id", "ub_wr", "bi_sc", "bi_rank", "bk"),
        ("genre", ctx["ug_top"], ctx["gipop"], "genre_id", "ug_wr", "gi_sc", "gi_rank", "gn")]:
        ut = etop[etop["user_id"].isin(user_ids)]
        if ut.empty: srcs[pre] = pd.DataFrame(columns=["user_id","edition_id"]); continue
        mg = ut.merge(eipop, on=ecol, how="inner")
        mg[f"{pre}_score"] = (mg[ewcol] * mg[escol]).astype("float32")
        agg = mg.groupby(["user_id","edition_id"], as_index=False).agg(**{
            f"{pre}_score":(f"{pre}_score","max"), f"{pre}_rank":(ercol,"min")})
        srcs[pre] = agg

    # Demo/gender/global pop
    ud = users_df[users_df["user_id"].isin(user_ids)][["user_id","gender","age_bucket"]]
    dm = ud.merge(ctx["dpop"], on=["gender","age_bucket"], how="left").dropna(subset=["edition_id"])
    gn = ud.merge(ctx["genp"], on=["gender"], how="left").dropna(subset=["edition_id"])
    if not dm.empty: dm["edition_id"] = dm["edition_id"].astype("int64")
    if not gn.empty: gn["edition_id"] = gn["edition_id"].astype("int64")
    srcs["demo"] = dm[["user_id","edition_id","d_sc","d_rank"]].copy() if not dm.empty else pd.DataFrame(columns=["user_id","edition_id"])
    srcs["gend"] = gn[["user_id","edition_id","g_sc","g_rank"]].copy() if not gn.empty else pd.DataFrame(columns=["user_id","edition_id"])
    gp = ctx["gpop"][["edition_id","pop","pop_rank"]].copy()
    if not gp.empty and len(user_ids) > 0:
        udf = pd.DataFrame({"user_id": user_ids.astype(np.int64), "_t": 1}); gp["_t"] = 1
        srcs["glob"] = udf.merge(gp, on="_t").drop(columns="_t")
    else: srcs["glob"] = pd.DataFrame(columns=["user_id","edition_id"])

    # Merge
    parts = [df[["user_id","edition_id"]] for df in srcs.values() if df is not None and not df.empty]
    if not parts: return pd.DataFrame(columns=["user_id","edition_id","pre_score"]), srcs
    pool = pd.concat(parts, ignore_index=True).drop_duplicates()
    for df in srcs.values():
        if df is None or df.empty: continue
        extra = [c for c in df.columns if c not in ("user_id","edition_id")]
        pool = pool.merge(df[["user_id","edition_id"]+extra], on=["user_id","edition_id"], how="left")
    pool = pool.merge(ctx["seen"].assign(_s=1), on=["user_id","edition_id"], how="left")
    pool = pool[pool["_s"].isna()].drop(columns="_s")

    # Pre-score
    sc = np.zeros(len(pool), dtype=np.float32)
    weights = {"af_rank": 2.8, "ar_rank": 2.2, "ax_rank": 2.0, "bpr_rank": 1.8,
               "lk_score": 1.6, "sk_score": 1.3, "ucf_score": 1.4, "au_rank": 1.1,
               "bk_rank": 0.9, "gn_rank": 0.8, "d_rank": 0.55, "g_rank": 0.35, "pop_rank": 0.45}
    for col, w in weights.items():
        if col in pool:
            if col.endswith("_score"): sc += w * pool[col].fillna(0).values.astype(np.float32)
            else: sc += w * r2s(pool[col].fillna(1e9)).values.astype(np.float32)
    if "cooc_score" in pool:
        mx = pool["cooc_score"].max()
        if mx > 0: sc += 1.5 * (pool["cooc_score"].fillna(0) / mx).values.astype(np.float32)
    rank_cols = [c for c in pool.columns if c.endswith("_rank")]
    pool["src_cnt"] = pool[rank_cols].notna().sum(axis=1).astype("int16") if rank_cols else 0
    sc += 0.15 * pool["src_cnt"].values.astype(np.float32)
    pool["pre_score"] = sc
    pool = pool.sort_values(["user_id","pre_score"], ascending=[True,False]).groupby("user_id").head(cfg.max_cands).reset_index(drop=True)
    return pool, srcs

# ═══════════════════════════════════════════════════════════════
# 8. FEATURE ASSEMBLY
# ═══════════════════════════════════════════════════════════════
def assemble_features(ctx, pool, hidden=None):
    df = pool.copy()
    df = df.merge(ctx["itf"], on="edition_id", how="left")
    df = df.merge(ctx["uf"], on="user_id", how="left")
    # Affinity
    for aff, cols, keys in [
        (ctx["ua"], ["ua_w","ua_wr","ua_n","ua_last","ua_rank"], ["user_id","author_id"]),
        (ctx["ub"], ["ub_w","ub_wr","ub_n","ub_last","ub_rank"], ["user_id","book_id"]),
        (ctx["ul"], ["ul_w","ul_wr","ul_n","ul_last","ul_rank"], ["user_id","language_id"]),
        (ctx["up"], ["up_w","up_wr","up_n","up_last","up_rank"], ["user_id","publisher_id"])]:
        if all(k in df.columns for k in keys):
            df = df.merge(aff[keys + cols], on=keys, how="left")
    # ALS pair features
    for key, pre in [("full","af"), ("rec","ar"), ("extra","ax")]:
        sc, co = ctx["als"][key].score_pairs(df["user_id"].values, df["edition_id"].values)
        df[f"{pre}_ps"] = sc; df[f"{pre}_pc"] = co
    if "bpr" in ctx["als"]:
        sc, co = ctx["als"]["bpr"].score_pairs(df["user_id"].values, df["edition_id"].values)
        df["bpr_ps"] = sc; df["bpr_pc"] = co
    else: df["bpr_ps"] = 0.0; df["bpr_pc"] = 0.0

    # Genre match (vectorized)
    try:
        utg = ctx["ug"].sort_values(["user_id","ug_wr"], ascending=[True,False]).groupby("user_id").head(5)[["user_id","genre_id"]]
        dg = df[["user_id","edition_id"]].merge(edition_genres[["edition_id","genre_id"]], on="edition_id", how="left")
        dg = dg[dg["genre_id"].notna()]
        dg = dg.merge(utg.assign(_m=1), on=["user_id","genre_id"], how="left")
        gm = dg.groupby(["user_id","edition_id"], as_index=False).agg(gm_cnt=("_m","sum"), gm_tot=("genre_id","size"))
        gm["gm_ratio"] = (gm["gm_cnt"].fillna(0) / np.maximum(gm["gm_tot"], 1)).astype("float32")
        df = df.merge(gm[["user_id","edition_id","gm_cnt","gm_ratio"]], on=["user_id","edition_id"], how="left")
    except: pass
    df["gm_cnt"] = df.get("gm_cnt", pd.Series(0, index=df.index)).fillna(0).astype("float32")
    df["gm_ratio"] = df.get("gm_ratio", pd.Series(0, index=df.index)).fillna(0).astype("float32")

    # Pre-incident velocity
    try:
        ow = ctx["obs_w"]; rt = ctx["ref"]
        pi = ow[(ow["event_ts"] >= rt - pd.Timedelta(days=44)) & (ow["event_ts"] < rt - pd.Timedelta(days=30))]
        if not pi.empty:
            pit = pi.groupby("edition_id", as_index=False).agg(pi_ev=("event_ts","size"), pi_us=("user_id","nunique"), pi_w=("w_full","sum"))
            df = df.merge(pit, on="edition_id", how="left")
    except: pass
    for c in ["pi_ev","pi_us","pi_w"]: df[c] = df.get(c, pd.Series(0, index=df.index)).fillna(0).astype("float32")
    df["pi_vel"] = (df["pi_w"] / np.maximum(df["i_w"], 1e-6)).astype("float32")

    # Cross features
    df["x_hot"] = (df["u_rec_ratio"].fillna(0) * df["i_hot"].fillna(0)).astype("float32")
    df["x_read"] = (df["u_read_sh"].fillna(0) * df["i_read_sh"].fillna(0)).astype("float32")
    df["x_trend"] = (df["u_trend"].fillna(0) * df["i_trend14"].fillna(0)).astype("float32")
    df["als_diff"] = (df["ar_ps"].fillna(0) - df["af_ps"].fillna(0)).astype("float32")
    df["als_cdiff"] = (df["ar_pc"].fillna(0) - df["af_pc"].fillna(0)).astype("float32")
    df["i_niche"] = (1.0 / np.maximum(df["i_users"].fillna(1), 1)).astype("float32")
    df["i_pop_ua"] = (df["i_events"].fillna(0) / np.maximum(df["u_events"].fillna(1), 1)).astype("float32")
    df["age_gap"] = (df["age"].fillna(-1) - df["age_restriction"].fillna(-1)).astype("float32")
    df["pub_gap"] = (2025 - df["publication_year"].fillna(2025)).astype("float32")

    # Fill
    num = df.select_dtypes(include=[np.number, "bool"]).columns
    for c in num: df[c] = df[c].fillna(0)
    cat_cols = ["author_id","book_id","language_id","publisher_id","gender","age_bucket"]
    for c in cat_cols:
        if c in df.columns: df[c] = df[c].fillna(-1).astype(str)

    if hidden is not None:
        df = df.merge(hidden.assign(label=1), on=["user_id","edition_id"], how="left")
        df["label"] = df["label"].fillna(0).astype("int8")
    return df

# ═══════════════════════════════════════════════════════════════
# 9. VALIDATION FOLDS
# ═══════════════════════════════════════════════════════════════
def make_folds(ints, cfg):
    mx = ints["event_ts"].max().normalize() + pd.Timedelta(days=1)
    mn = ints["event_ts"].min().normalize()
    folds = []
    for i in range(cfg.n_pseudo_folds):
        end = mx - pd.Timedelta(days=cfg.fold_days * (cfg.n_pseudo_folds - 1 - i))
        start = end - pd.Timedelta(days=cfg.fold_days)
        if start <= mn: continue
        folds.append((f"f{i}", start, end, cfg.seed + 100 + i))
    return folds

def build_masked(ints, start, end, hide_rate, seed):
    before = ints[ints["event_ts"] < start][["user_id","edition_id"]].drop_duplicates()
    inc = ints[(ints["event_ts"] >= start) & (ints["event_ts"] < end)].copy()
    elig = inc.merge(before.assign(_s=1), on=["user_id","edition_id"], how="left")
    elig = elig[elig["_s"].isna()].drop(columns="_s")
    if elig.empty: return ints[ints["event_ts"] < end].copy(), pd.DataFrame(columns=["user_id","edition_id"]), np.array([], dtype=np.int64)
    ep = elig.groupby(["user_id","edition_id"], as_index=False).agg(cnt=("event_ts","size"), last=("event_ts","max"), hr=("event_type", lambda x: int((x==2).any())))
    rng = np.random.default_rng(seed)
    hide_idx = []
    for uid, grp in ep.groupby("user_id"):
        n = len(grp); nr = float(np.clip(hide_rate * rng.lognormal(0, 0.35), 0.05, 0.6))
        nh = max(0, min(int(round(n * nr)), n))
        if nh == 0 and n >= 3 and rng.random() < 0.35: nh = 1
        if nh == 0: continue
        p = (1 + 0.5*grp["hr"].values + 0.25*np.log1p(grp["cnt"].values)).astype(np.float64)
        p /= p.sum()
        hide_idx.extend(rng.choice(grp.index.values, size=nh, replace=False, p=p).tolist())
    hidden = ep.loc[hide_idx, ["user_id","edition_id"]].drop_duplicates().reset_index(drop=True)
    obs = ints[ints["event_ts"] < end].copy()
    if not hidden.empty:
        obs = obs.merge(hidden.assign(_h=1), on=["user_id","edition_id"], how="left")
        obs = obs[obs["_h"].isna()].drop(columns="_h")
    fold_users = np.array(sorted(hidden["user_id"].unique()), dtype=np.int64)
    return obs.reset_index(drop=True), hidden, fold_users

def prepare_fold(ints, name, start, end, seed):
    obs, hidden, fusers = build_masked(ints, start, end, cfg.hide_rate, seed)
    ctx = build_context(obs, end)
    pool, _ = gen_candidates(ctx, fusers)
    r50 = recall_at_k(pool, hidden, 50); r200 = recall_at_k(pool, hidden, 200)
    print(f"[{name}] recall@50={r50:.4f} @200={r200:.4f}")
    # Inject missed positives
    miss = hidden.merge(pool[["user_id","edition_id"]], on=["user_id","edition_id"], how="left", indicator=True)
    miss = miss[miss["_merge"]=="left_only"][["user_id","edition_id"]]; miss["pre_score"] = 0.0
    if not miss.empty: pool = pd.concat([pool, miss], ignore_index=True, sort=False).drop_duplicates(["user_id","edition_id"])
    feat = assemble_features(ctx, pool, hidden)
    base = eval_scored(feat, hidden, "pre_score")
    print(f"[{name}] heuristic NDCG@20={base:.5f}")
    del ctx; gc.collect()
    return {"hidden": hidden, "users": fusers, "feat": feat, "ndcg": base}

print("Building folds...")
folds = make_folds(interactions, cfg)
for f in folds: print(f"  {f[0]}: {f[1]} -> {f[2]}")
fold_arts = []
for name, start, end, seed in folds:
    print("="*80)
    art = prepare_fold(interactions, name, start, end, seed)
    if art["feat"].empty or art["hidden"].empty: print(f"Skip {name}"); continue
    fold_arts.append(art)
    gc.collect()

# ═══════════════════════════════════════════════════════════════
# 10. CATBOOST TRAINING
# ═══════════════════════════════════════════════════════════════
from catboost import CatBoostRanker, Pool as CBPool

def downsample_neg(df, max_neg, seed):
    if "label" not in df.columns: return df
    rng = np.random.default_rng(seed)
    pos = df[df["label"]==1]; neg = df[df["label"]==0]
    if neg.empty: return pos
    neg = neg.sort_values(["user_id","pre_score"], ascending=[True,False])
    hard = neg.groupby("user_id").head(max_neg // 2)
    rest = neg.merge(hard[["user_id","edition_id"]].assign(_h=1), on=["user_id","edition_id"], how="left")
    rest = rest[rest["_h"].isna()].drop(columns="_h")
    sampled = []
    nr = max_neg - max_neg // 2
    for _, grp in rest.groupby("user_id"):
        if len(grp) <= nr: sampled.append(grp)
        else: sampled.append(grp.loc[rng.choice(grp.index.values, nr, replace=False)])
    return pd.concat([pos, hard] + sampled, ignore_index=True).sort_values(["user_id","pre_score"], ascending=[True,False]).reset_index(drop=True)

assert len(fold_arts) >= 2, "Need >= 2 folds"
val_art = fold_arts[-1]; train_arts = fold_arts[:-1]
train_df = pd.concat([a["feat"] for a in train_arts], ignore_index=True)
valid_df = val_art["feat"].copy()
train_df = downsample_neg(train_df, cfg.max_neg, cfg.seed)

gs = train_df.groupby("user_id")["edition_id"].size()
gp = train_df.groupby("user_id")["label"].sum()
good = set(gs[gs >= cfg.min_group].index) & set(gp[gp > 0].index)
train_df = train_df[train_df["user_id"].isin(good)].copy()

DROP = {"label","fold_name"}; KEYS = ["user_id","edition_id"]
CAT = ["author_id","book_id","language_id","publisher_id","gender","age_bucket"]
feat_cols = [c for c in train_df.columns if c not in DROP and c not in KEYS]
cat_cols = [c for c in CAT if c in feat_cols]
num_cols = [c for c in feat_cols if c not in cat_cols]
for c in num_cols:
    train_df[c] = pd.to_numeric(train_df[c], errors="coerce").fillna(0).astype("float32")
    valid_df[c] = pd.to_numeric(valid_df[c], errors="coerce").fillna(0).astype("float32")
for c in cat_cols:
    train_df[c] = train_df[c].fillna("UNK").astype(str)
    valid_df[c] = valid_df[c].fillna("UNK").astype(str)

tp = CBPool(train_df[feat_cols], train_df["label"], group_id=train_df["user_id"], cat_features=cat_cols)
vp = CBPool(valid_df[feat_cols], valid_df["label"], group_id=valid_df["user_id"], cat_features=cat_cols)
tt = "GPU" if cfg.use_gpu else "CPU"

loss_a = "YetiRankPairwise" if tt == "GPU" else "YetiRank"
pa = dict(loss_function=loss_a, eval_metric="NDCG:top=20", task_type=tt, random_seed=cfg.seed,
    iterations=cfg.cb_iter, learning_rate=cfg.cb_lr, depth=8 if tt=="GPU" else 10,
    l2_leaf_reg=8, bootstrap_type="Bernoulli", subsample=0.8, verbose=200,
    od_type="Iter", od_wait=200, border_count=32 if tt=="GPU" else 254)
pb = dict(loss_function="QuerySoftMax", eval_metric="NDCG:top=20", task_type=tt, random_seed=cfg.seed+17,
    iterations=cfg.cb_iter_b, learning_rate=cfg.cb_lr_b, depth=10 if tt=="CPU" else 8,
    l2_leaf_reg=6, bootstrap_type="Bernoulli", subsample=0.85, verbose=200,
    od_type="Iter", od_wait=200, border_count=128 if tt=="GPU" else 254)
loss_c = "PairLogitPairwise" if tt == "GPU" else "PairLogit"
pc = dict(loss_function=loss_c, eval_metric="NDCG:top=20", task_type=tt, random_seed=cfg.seed+31,
    iterations=cfg.cb_iter_c, learning_rate=cfg.cb_lr_c, depth=8, l2_leaf_reg=10,
    bootstrap_type="Bernoulli", subsample=0.8, verbose=200,
    od_type="Iter", od_wait=200, border_count=32 if tt=="GPU" else 254)

ma = CatBoostRanker(**pa); mb = CatBoostRanker(**pb); mc = CatBoostRanker(**pc)
ma.fit(tp, eval_set=vp, use_best_model=True)
mb.fit(tp, eval_set=vp, use_best_model=True)
mc.fit(tp, eval_set=vp, use_best_model=True)

va, vb, vc = ma.predict(vp), mb.predict(vp), mc.predict(vp)
ve = valid_df[["user_id","edition_id"]].copy()
ve["pa"], ve["pb"], ve["pc"] = va, vb, vc; ve["pre"] = valid_df["pre_score"].values

sa = eval_scored(ve.rename(columns={"pa":"score"}), val_art["hidden"], "score")
sb = eval_scored(ve.rename(columns={"pb":"score"}), val_art["hidden"], "score")
sc_val = eval_scored(ve.rename(columns={"pc":"score"}), val_art["hidden"], "score")
print(f"Val NDCG@20: YetiRank={sa:.6f} QuerySoftMax={sb:.6f} PairLogit={sc_val:.6f}")

# Blend search
bw, bs = (0.33,0.33,0.34), -1
for wa in np.linspace(0,1,11):
    for wb in np.linspace(0, 1-wa, max(2, int(11*(1-wa)+0.5))):
        wc = max(1-wa-wb, 0)
        t = ve[["user_id","edition_id"]].copy()
        t["score"] = (wa*ve["pa"] + wb*ve["pb"] + wc*ve["pc"]).astype(np.float32)
        s = eval_scored(t, val_art["hidden"], "score")
        if s > bs: bs, bw = s, (float(wa), float(wb), float(wc))
print(f"Best blend: {bw}, NDCG={bs:.6f}")

# Pre-score blend
ba, bsc = 0.0, bs
for alpha in np.linspace(0, 0.3, 16):
    t = ve[["user_id","edition_id"]].copy()
    ms = bw[0]*ve["pa"] + bw[1]*ve["pb"] + bw[2]*ve["pc"]
    pn = ve["pre"] / max(ve["pre"].std(), 1e-6) * ms.std()
    t["score"] = ((1-alpha)*ms + alpha*pn).astype(np.float32)
    s = eval_scored(t, val_art["hidden"], "score")
    if s > bsc: bsc, ba = s, float(alpha)
print(f"Pre-alpha={ba:.3f}, NDCG={bsc:.6f}")

# Stacking
from sklearn.linear_model import LogisticRegression
use_stack = False; stack_m = None
try:
    sf = np.column_stack([ve["pa"].values, ve["pb"].values, ve["pc"].values, ve["pre"].values])
    sl = valid_df["label"].values
    stack_m = LogisticRegression(C=1.0, max_iter=500, random_state=cfg.seed).fit(sf, sl)
    sp = stack_m.predict_proba(sf)[:, 1]
    t = ve[["user_id","edition_id"]].copy(); t["score"] = sp.astype(np.float32)
    ss = eval_scored(t, val_art["hidden"], "score")
    print(f"Stacking NDCG={ss:.6f}")
    use_stack = ss > bsc
    if use_stack: print(">>> Using stacking")
    else: print(">>> Using blend")
except Exception as e: print(f"Stack fail: {e}")

# ═══════════════════════════════════════════════════════════════
# 11. RETRAIN ON ALL FOLDS
# ═══════════════════════════════════════════════════════════════
all_df = pd.concat([a["feat"] for a in fold_arts], ignore_index=True)
all_df = downsample_neg(all_df, cfg.max_neg, cfg.seed+123)
gs = all_df.groupby("user_id")["edition_id"].size(); gp = all_df.groupby("user_id")["label"].sum()
good = set(gs[gs >= cfg.min_group].index) & set(gp[gp > 0].index)
all_df = all_df[all_df["user_id"].isin(good)].copy()
for c in num_cols: all_df[c] = pd.to_numeric(all_df[c], errors="coerce").fillna(0).astype("float32")
for c in cat_cols: all_df[c] = all_df[c].fillna("UNK").astype(str)
fp = CBPool(all_df[feat_cols], all_df["label"], group_id=all_df["user_id"], cat_features=cat_cols)

for m, p in [(ma, pa), (mb, pb), (mc, pc)]:
    bi = m.get_best_iteration()
    p2 = p.copy(); p2["iterations"] = p["iterations"] if bi is None or bi <= 0 else int(bi * 1.08)
    m.__class__(**p2).fit(fp, verbose=200)

fma = CatBoostRanker(**{**pa, "iterations": int((ma.get_best_iteration() or pa["iterations"]) * 1.08)})
fmb = CatBoostRanker(**{**pb, "iterations": int((mb.get_best_iteration() or pb["iterations"]) * 1.08)})
fmc = CatBoostRanker(**{**pc, "iterations": int((mc.get_best_iteration() or pc["iterations"]) * 1.08)})
fma.fit(fp, verbose=200); fmb.fit(fp, verbose=200); fmc.fit(fp, verbose=200)

if use_stack:
    ap = fma.predict(fp); bp2 = fmb.predict(fp); cp = fmc.predict(fp)
    saf = np.column_stack([ap, bp2, cp, all_df["pre_score"].values])
    stack_m = LogisticRegression(C=1.0, max_iter=500, random_state=cfg.seed).fit(saf, all_df["label"].values)

# ═══════════════════════════════════════════════════════════════
# 12. TEST INFERENCE
# ═══════════════════════════════════════════════════════════════
print("Building test context...")
full_ref = interactions["event_ts"].max().normalize() + pd.Timedelta(days=1)
full_ctx = build_context(interactions, full_ref)
target_uids = targets["user_id"].drop_duplicates().sort_values().values.astype(np.int64)
test_pool, _ = gen_candidates(full_ctx, target_uids)
test_feat = assemble_features(full_ctx, test_pool, hidden=None)

for c in num_cols:
    if c in test_feat.columns: test_feat[c] = pd.to_numeric(test_feat[c], errors="coerce").fillna(0).astype("float32")
    else: test_feat[c] = 0.0
for c in cat_cols:
    if c in test_feat.columns: test_feat[c] = test_feat[c].fillna("UNK").astype(str)
    else: test_feat[c] = "UNK"
for c in feat_cols:
    if c not in test_feat.columns: test_feat[c] = 0.0 if c in num_cols else "UNK"

tpool = CBPool(test_feat[feat_cols], cat_features=cat_cols)
tpa, tpb, tpc = fma.predict(tpool), fmb.predict(tpool), fmc.predict(tpool)

scored = test_feat[["user_id","edition_id","pre_score"]].copy()
scored["pa"], scored["pb"], scored["pc"] = tpa.astype(np.float32), tpb.astype(np.float32), tpc.astype(np.float32)

if use_stack and stack_m is not None:
    sf = np.column_stack([scored["pa"].values, scored["pb"].values, scored["pc"].values, scored["pre_score"].values])
    scored["final_score"] = stack_m.predict_proba(sf)[:, 1].astype(np.float32)
else:
    wa, wb, wc = bw
    ms = wa*scored["pa"] + wb*scored["pb"] + wc*scored["pc"]
    if ba > 0:
        pn = scored["pre_score"] / max(scored["pre_score"].std(), 1e-6) * ms.std()
        scored["final_score"] = ((1-ba)*ms + ba*pn).astype(np.float32)
    else:
        scored["final_score"] = ms.astype(np.float32)

# ═══════════════════════════════════════════════════════════════
# 13. SUBMISSION WITH MMR
# ═══════════════════════════════════════════════════════════════
print("Building submission...")
item_author = edition_meta.set_index("edition_id")["author_id"].to_dict()
seen_map = defaultdict(set)
for r in full_ctx["seen"].itertuples(index=False): seen_map[int(r.user_id)].add(int(r.edition_id))

# Fallback
ud = users_df[["user_id","gender","age_bucket"]]
dm_map, gn_map = defaultdict(list), defaultdict(list)
for r in full_ctx["dpop"].sort_values(["gender","age_bucket","d_rank"]).itertuples(index=False):
    dm_map[(int(r.gender), int(r.age_bucket))].append(int(r.edition_id))
for r in full_ctx["genp"].sort_values(["gender","g_rank"]).itertuples(index=False):
    gn_map[int(r.gender)].append(int(r.edition_id))
gl = full_ctx["gpop"].sort_values("pop_rank")["edition_id"].astype(int).tolist()

fb = {}
for r in ud[ud["user_id"].isin(target_uids)].itertuples(index=False):
    seq = dm_map.get((int(r.gender), int(r.age_bucket)), []) + gn_map.get(int(r.gender), []) + gl
    u = []; s = set()
    for x in seq:
        if x not in s: u.append(x); s.add(x)
    fb[int(r.user_id)] = u
for uid in target_uids:
    if int(uid) not in fb: fb[int(uid)] = gl[:]

# Build submission with MMR
scored = scored.sort_values(["user_id","final_score"], ascending=[True, False])
grouped = {int(u): grp["edition_id"].astype(int).tolist() for u, grp in scored.groupby("user_id")}
grouped_scores = {int(u): grp["final_score"].tolist() for u, grp in scored.groupby("user_id")}

sub_rows = []
for uid in tqdm(target_uids.astype(int), desc="Submission"):
    recs, used = [], set()
    seen = seen_map.get(uid, set())
    cands = grouped.get(uid, [])
    cand_sc = grouped_scores.get(uid, [])
    # Filter seen/used first
    valid_cands = [(eid, sc) for eid, sc in zip(cands, cand_sc) if eid not in seen]
    if len(valid_cands) <= TOP_K:
        for eid, sc in valid_cands:
            if eid not in used: recs.append(eid); used.add(eid)
    else:
        # MMR-like: greedily select, penalizing same author
        remaining = list(valid_cands[:min(len(valid_cands), TOP_K * 5)])  # top candidates
        sel_authors = []
        for _ in range(TOP_K):
            if not remaining: break
            best_idx, best_adj = 0, -1e18
            for j, (eid, sc) in enumerate(remaining):
                a = item_author.get(eid, -1)
                pen = sum(0.12 for sa in sel_authors if sa == a and a != -1)
                adj = sc - pen
                if adj > best_adj: best_adj, best_idx = adj, j
            eid, sc = remaining.pop(best_idx)
            recs.append(eid); used.add(eid)
            sel_authors.append(item_author.get(eid, -1))
    # Fallback
    if len(recs) < TOP_K:
        for eid in fb.get(uid, []):
            if eid not in used and eid not in seen:
                recs.append(eid); used.add(eid)
                if len(recs) == TOP_K: break
    if len(recs) < TOP_K:
        for eid in edition_meta["edition_id"].astype(int).tolist():
            if eid not in used and eid not in seen:
                recs.append(eid); used.add(eid)
                if len(recs) == TOP_K: break
    for rank, eid in enumerate(recs[:TOP_K], 1):
        sub_rows.append({"user_id": uid, "edition_id": eid, "rank": rank})

submission = pd.DataFrame(sub_rows)
assert len(submission) == len(target_uids) * TOP_K, f"Expected {len(target_uids)*TOP_K}, got {len(submission)}"
submission.to_csv(cfg.output_dir / "submission.csv", index=False)
print(f"Saved submission: {submission.shape}")
print(submission.head(20))
print("DONE!")

Data root: /kaggle/input/datasets/artemnazemtsev/nto-ai
Loading data...
  interactions: (443278, 5)
  targets: (3862, 1)
  editions: (460599, 11)
  users: (80738, 4)
implicit: True
Building folds...
  f0: 2025-08-03 00:00:00 -> 2025-09-02 00:00:00
  f1: 2025-09-02 00:00:00 -> 2025-10-02 00:00:00
  f2: 2025-10-02 00:00:00 -> 2025-11-01 00:00:00
  f3: 2025-11-01 00:00:00 -> 2025-12-01 00:00:00


  0%|          | 0/16 [00:00<?, ?it/s]

  ALS fitted: users=(15020, 192), items=(87375, 192), matrix=(15020, 87375)


  0%|          | 0/12 [00:00<?, ?it/s]

  ALS fitted: users=(15020, 128), items=(87375, 128), matrix=(15020, 87375)


...

  ALS fitted: users=(19259, 128), items=(123671, 128), matrix=(19259, 123671)


  0%|          | 0/14 [00:00<?, ?it/s]

  ALS fitted: users=(19259, 256), items=(123671, 256), matrix=(19259, 123671)


  0%|          | 0/100 [00:00<?, ?it/s]

BPR OK
[f3] recall@50=0.0834 @200=0.1273
